# Task 2C: Evaluation and baseline comparison
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krishanSKDA/CDAZZDEV-MLE-Krishan_Danushka/blob/main/task2_genai/Task2_Eval.ipynb)

Base model (same system prompt, no fine-tuning) vs merged fine-tuned model on the **identical** held-out test set, greedy decoding for both. Needs a T4 GPU and `GEMINI_API_KEY` (LLM judge: `gemini-3.1-flash-lite`).

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/krishanSKDA/CDAZZDEV-MLE-Krishan_Danushka.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "task2_genai/requirements-train.txt"], check=True)
else:
    while not os.path.exists("requirements.txt"):
        os.chdir("..")
sys.path.insert(0, os.getcwd())
print("repo root:", os.getcwd(), "| colab:", IN_COLAB)

In [ ]:
# Colab ships torchao 0.10, which recent peft/transformers reject; this project does not use torchao
!pip uninstall -y -q torchao

In [ ]:
import gc, json, torch, pandas as pd
from task2_genai.dataset import DATA_DIR, load_jsonl
from task2_genai.finetune import TrainConfig, free_gpu
from task2_genai.evaluation import generate, load_for_inference, prompt_messages

BASE_ID = TrainConfig().base_model
from common.config import get_secret
from huggingface_hub import HfApi
FT_ID = f"{HfApi(token=get_secret('HF_TOKEN')).whoami()['name']}/qwen2.5-3b-risk-extractor"  # pushed by the fine-tune notebook
test = load_jsonl(DATA_DIR / "test.jsonl")
os.makedirs("task2_genai/results", exist_ok=True)
print(len(test), "test examples")

In [ ]:
def run(model_id, name):
    model, tok = load_for_inference(model_id)
    gens = generate(model, tok, [prompt_messages(r) for r in test])
    with open(f"task2_genai/results/{name}_generations.jsonl", "w", encoding="utf-8") as f:
        for r, g in zip(test, gens):
            f.write(json.dumps({"id": r["id"], **g}, ensure_ascii=False) + "\n")
    del model
    free_gpu()
    return gens

base_gens = run(BASE_ID, "base")
ft_gens = run(FT_ID, "finetuned")

## ROUGE-L and structural metrics

In [ ]:
from task2_genai.evaluation import evaluate_model
base_summary, base_rows = evaluate_model("base (system prompt only)", test, base_gens)
ft_summary, ft_rows = evaluate_model("fine-tuned (QLoRA)", test, ft_gens)
comparison = pd.DataFrame([base_summary, ft_summary]).set_index("model").T
comparison["delta"] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
comparison.round(4)

## Additional metrics: BERTScore F1 and LLM-as-judge (structured JSON rubric)

In [ ]:
from task2_genai.evaluation import bertscore_f1, gold_of
refs = [json.dumps(gold_of(r), ensure_ascii=False) for r in test]
try:
    base_rows["bertscore_f1"] = bertscore_f1([g["text"] for g in base_gens], refs)
    ft_rows["bertscore_f1"] = bertscore_f1([g["text"] for g in ft_gens], refs)
    comparison.loc["bertscore_f1"] = [base_rows["bertscore_f1"].mean(), ft_rows["bertscore_f1"].mean(),
                                      ft_rows["bertscore_f1"].mean() - base_rows["bertscore_f1"].mean()]
except Exception as exc:  # library incompatibility must not block the LLM-judge metric below
    print(f"BERTScore unavailable: {type(exc).__name__}: {exc}")
comparison.round(4)

In [ ]:
from task2_genai.evaluation import judge_all, JUDGE_MODEL
print("judge:", JUDGE_MODEL)
judge_base = judge_all(test, [g["text"] for g in base_gens])
judge_ft = judge_all(test, [g["text"] for g in ft_gens])
crit = ["faithfulness", "completeness", "severity_calibration", "format_compliance"]
judge_table = pd.DataFrame({"base": judge_base[crit].mean(), "fine-tuned": judge_ft[crit].mean()})
judge_table.loc["verdict=correct"] = [(judge_base.verdict == "correct").mean(), (judge_ft.verdict == "correct").mean()]
judge_table.loc["verdict=hallucinated"] = [(judge_base.verdict == "hallucinated").mean(), (judge_ft.verdict == "hallucinated").mean()]
judge_table.round(3)

In [ ]:
judge_ft.head(5)

In [ ]:
comparison.to_csv("task2_genai/results/metrics_comparison.csv")
judge_table.to_csv("task2_genai/results/judge_summary.csv")
base_rows.merge(judge_base, on="id").to_csv("task2_genai/results/base_per_sample.csv", index=False)
ft_rows.merge(judge_ft, on="id").to_csv("task2_genai/results/finetuned_per_sample.csv", index=False)

## Manual review: fine-tuned model (≥ 10 responses)
Each response below shows the passage, reference and prediction, with an *automatic* pre-check (quote grounding + category match). **Read each one yourself** and record your label (`correct`, `partially_correct`, `hallucinated`) in `MANUAL_LABELS`; the auto-label is only a hint.

In [ ]:
from task2_genai.evaluation import review_sheet
N_REVIEW = len(test)
sheet = review_sheet(test, [g["text"] for g in ft_gens], ft_rows, N_REVIEW)
for _, r in sheet.iterrows():
    print("=" * 100, f"\n{r.id}  auto={r.auto_label} ({r.auto_reason})\n")
    print("PASSAGE:\n", r.passage, "\n\nREFERENCE:\n", r.gold, "\n\nPREDICTION:\n", r.prediction)

print("\nCopy into the next cell and fill each label with correct / partially_correct / hallucinated:")
print("MANUAL_LABELS = {\n" + "".join(f'    "{i}": "",\n' for i in sheet.id) + "}")

In [ ]:
# Fill in after reading every response above: {"ex_0012": "correct", ...}
MANUAL_LABELS = {
}

from task2_genai.evaluation import hallucination_rate
if len(MANUAL_LABELS) >= 10:
    result = hallucination_rate(MANUAL_LABELS)
    print("MANUAL REVIEW:", result)
    sheet["manual_label"] = sheet["id"].map(MANUAL_LABELS)
    sheet[["id", "auto_label", "manual_label"]].to_csv("task2_genai/results/manual_review.csv", index=False)
else:
    print("Manual labels not filled yet. Automatic pre-check for reference:", hallucination_rate(dict(zip(sheet.id, sheet.auto_label))))

## Qualitative analysis

*Write two paragraphs after reviewing the results above, citing example ids.*

**Where fine-tuning helped.** …

**Remaining failure modes and next steps.** …

## Bonus: RAG fallback (perplexity-gated ChromaDB retrieval)
Confidence = perplexity of the model's own generated tokens. The threshold is the 80th percentile on the validation set; above it (or on invalid JSON) the two most similar verified training extractions are retrieved from ChromaDB and the model is re-queried with them as calibration context.

In [ ]:
from task2_genai.rag_fallback import answer_with_fallback, build_store, calibrate_threshold
from task2_genai.domain import label_response
from task2_genai.evaluation import passage_of
train, val = load_jsonl(DATA_DIR / "train.jsonl"), load_jsonl(DATA_DIR / "val.jsonl")
store = build_store(train)
ft_model, ft_tok = load_for_inference(FT_ID)
val_gens = generate(ft_model, ft_tok, [prompt_messages(r) for r in val])
THRESHOLD = calibrate_threshold([g["perplexity"] for g in val_gens])
print(f"perplexity threshold: {THRESHOLD:.4f}")

In [ ]:
ranked = sorted(zip(test, ft_gens), key=lambda x: -x[1]["perplexity"])
cases = [answer_with_fallback(ft_model, ft_tok, rec, store, THRESHOLD) for rec, _ in ranked[:3]]
for rec, case in zip([r for r, _ in ranked[:3]], cases):
    print("=" * 100, f"\n{case['id']} triggered={case['triggered']} reason={case['trigger_reason']}")
    print("BEFORE:", label_response(passage_of(rec), gold_of(rec), case["first"]["text"]), "\n", case["first"]["text"])
    if case["triggered"]:
        print("RETRIEVED:", case["retrieved"])
        print("AFTER: ", label_response(passage_of(rec), gold_of(rec), case["second"]["text"]), "\n", case["second"]["text"])